# Distributed Training at Scale — Walkthrough

This notebook covers the core concepts and tools for distributed training:

1. Parallelism strategies: FSDP, pipeline, and tensor parallelism
2. Memory calculations for ZeRO stages
3. Running a simple FSDP training example (single-process mode)
4. Profiling throughput

Everything runs in single-process CPU mode — no multi-GPU setup required.

In [ ]:
import sys
sys.path.insert(0, "..")

import time
import torch
import torch.nn as nn
import numpy as np

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not installed — plots will be skipped")

from src.fsdp.trainer import SimpleTransformer, TrainConfig
from src.fsdp.sharding import FSDPConfig, ShardMode, PrecisionMode, get_sharding_strategy
from src.profiling.throughput import (
    ThroughputTracker,
    compute_scaling_efficiency,
    compute_tflops,
    compute_mfu,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Parallelism Strategies Explained

Modern models don't fit on a single GPU. Three complementary strategies split the work:

| Strategy | What's split | Communication | Best for |
|----------|-------------|---------------|----------|
| **FSDP / ZeRO** | Parameters, gradients, optimizer states | All-gather + reduce-scatter per layer | 1B–30B params, minimal code changes |
| **Pipeline** | Model layers across GPUs | Point-to-point between adjacent stages | Models too large for one node |
| **Tensor** | Individual weight matrices | All-reduce per layer | Within-node with NVLink |

In practice, large-scale training combines all three:
- Tensor parallelism within a node (fast NVLink)
- Pipeline parallelism across nodes in a rack
- Data parallelism (FSDP) across racks

### Pipeline Bubble Overhead

Pipeline parallelism introduces idle time (the "bubble"). With $S$ stages and $M$ micro-batches:

$$\text{bubble fraction} = \frac{S - 1}{S + M - 1}$$

Let's compute this for different configurations:

In [ ]:
def pipeline_bubble_fraction(stages: int, microbatches: int) -> float:
    """Compute the fraction of time GPUs sit idle in a pipeline."""
    return (stages - 1) / (stages + microbatches - 1)

print("Pipeline Bubble Overhead")
print(f"{'Stages':>8} {'μ-batches':>10} {'Bubble':>10} {'Efficiency':>12}")
print("-" * 44)
for stages in [2, 4, 8]:
    for mb in [4, 8, 16, 32]:
        bubble = pipeline_bubble_fraction(stages, mb)
        print(f"{stages:>8} {mb:>10} {bubble:>9.1%} {1 - bubble:>11.1%}")

## 2. ZeRO Memory Calculations

Standard DDP replicates everything on every GPU. ZeRO eliminates this redundancy in three stages:

- **Stage 1**: Shard optimizer states → each GPU stores $4P + 12P/N$ bytes
- **Stage 2**: Also shard gradients → each GPU stores $2P + 14P/N$ bytes
- **Stage 3**: Shard everything → each GPU stores $16P/N$ bytes

where $P$ = parameters (in fp16 bytes) and $N$ = number of GPUs.

In [ ]:
def zero_memory_per_gpu(num_params_billions: float, num_gpus: int) -> dict:
    """Calculate memory per GPU for each ZeRO stage.

    Args:
        num_params_billions: Model size in billions of parameters.
        num_gpus: Number of GPUs in the cluster.

    Returns:
        Dict mapping strategy name to memory in GB.
    """
    P = num_params_billions * 1e9  # number of parameters
    N = num_gpus

    # Memory in bytes, then convert to GB
    ddp = 16 * P  # full replication: 2P (fp16 params) + 2P (fp16 grads) + 12P (fp32 optimizer)
    zero1 = 4 * P + 12 * P / N  # shard optimizer states
    zero2 = 2 * P + 14 * P / N  # also shard gradients
    zero3 = 16 * P / N          # shard everything

    to_gb = lambda b: b / (1024 ** 3)
    return {
        "DDP (full replication)": to_gb(ddp),
        "ZeRO Stage 1 (shard optimizer)": to_gb(zero1),
        "ZeRO Stage 2 (+ shard grads)": to_gb(zero2),
        "ZeRO Stage 3 / FSDP (shard all)": to_gb(zero3),
    }


# Calculate for a 7B model across different GPU counts
for num_gpus in [1, 4, 8, 64]:
    print(f"\n=== 7B model on {num_gpus} GPU(s) ===")
    mem = zero_memory_per_gpu(7.0, num_gpus)
    for strategy, gb in mem.items():
        fits = "✓" if gb < 80 else "✗ (>80GB A100)"
        print(f"  {strategy:<40s} {gb:>8.1f} GB  {fits}")

In [ ]:
# Visualize memory savings across GPU counts
if HAS_MPL:
    gpu_counts = [1, 2, 4, 8, 16, 32, 64]
    model_size = 7.0  # 7B params

    strategies = {}
    for n in gpu_counts:
        mem = zero_memory_per_gpu(model_size, n)
        for name, gb in mem.items():
            strategies.setdefault(name, []).append(gb)

    fig, ax = plt.subplots(figsize=(10, 5))
    for name, values in strategies.items():
        ax.plot(gpu_counts, values, "o-", label=name, linewidth=2)

    ax.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="A100 80GB limit")
    ax.set_xlabel("Number of GPUs")
    ax.set_ylabel("Memory per GPU (GB)")
    ax.set_title(f"ZeRO Memory Savings — {model_size:.0f}B Parameter Model")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(gpu_counts)
    ax.set_xticklabels([str(n) for n in gpu_counts])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("zero_memory.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved zero_memory.png")
else:
    print("Install matplotlib to see the memory chart.")

## 3. FSDP Training Example (Single Process)

We run a minimal training loop using the project's `SimpleTransformer` model.
In single-process mode, FSDP wrapping is skipped (it requires `dist.init_process_group`),
but the training loop, optimizer, and loss computation are identical to the
multi-GPU version.

In [ ]:
# Build a small transformer
train_config = TrainConfig(
    vocab_size=512,
    d_model=128,
    n_heads=4,
    n_layers=2,
    seq_len=64,
    batch_size=8,
    lr=3e-4,
    max_steps=40,
    log_interval=10,
)

model = SimpleTransformer(
    vocab_size=train_config.vocab_size,
    d_model=train_config.d_model,
    n_heads=train_config.n_heads,
    n_layers=train_config.n_layers,
    seq_len=train_config.seq_len,
)

param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count:,}")

# Synthetic data: random token sequences
num_samples = train_config.max_steps * train_config.batch_size * 2
data = torch.randint(0, train_config.vocab_size, (num_samples, train_config.seq_len))

optimizer = torch.optim.AdamW(model.parameters(), lr=train_config.lr, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

# Training loop
model.train()
losses = []
step = 0

for i in range(0, len(data) - train_config.batch_size, train_config.batch_size):
    if step >= train_config.max_steps:
        break

    batch = data[i : i + train_config.batch_size]
    inputs = batch[:, :-1]
    targets = batch[:, 1:]

    logits = model(inputs)
    loss = loss_fn(logits.reshape(-1, train_config.vocab_size), targets.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())
    step += 1

    if step % train_config.log_interval == 0:
        print(f"Step {step:3d}/{train_config.max_steps} | Loss: {loss.item():.4f}")

print(f"\nTraining complete: {step} steps, final loss {losses[-1]:.4f}")

## 4. Profiling Throughput

The `ThroughputTracker` measures tokens/sec, samples/sec, and step latency.
We also compute scaling efficiency and Model FLOPS Utilization (MFU) to
understand how well hardware is being used.

In [ ]:
# Profile the trained model's throughput
tracker = ThroughputTracker(world_size=1, log_interval=5)
model.eval()

num_profile_steps = 30
batch_size = 8
seq_len = train_config.seq_len - 1  # inputs are seq_len - 1

with torch.no_grad():
    for i in range(num_profile_steps):
        inputs = torch.randint(0, train_config.vocab_size, (batch_size, seq_len))

        tracker.start_step()
        _ = model(inputs)
        metrics = tracker.end_step(
            tokens=batch_size * seq_len,
            samples=batch_size,
        )

stats = tracker.get_stats()
print(f"Throughput Summary ({num_profile_steps} steps):")
print(f"  Tokens/sec:     {stats.tokens_per_sec:>12,.0f}")
print(f"  Samples/sec:    {stats.samples_per_sec:>12,.1f}")
print(f"  Avg step time:  {stats.avg_step_time_sec * 1000:>12.1f} ms")
print(f"  Peak tokens/s:  {stats.peak_tokens_per_sec:>12,.0f}")

# Estimate TFLOPS
tflops = compute_tflops(param_count, stats.tokens_per_sec, seq_len, batch_size)
print(f"  Est. TFLOPS:    {tflops:>12.3f}")

In [ ]:
# Simulate scaling efficiency from hypothetical multi-GPU throughputs
# These are illustrative numbers for a typical FSDP training run
hypothetical_throughputs = {
    1: 50_000,
    2: 92_000,
    4: 175_000,
    8: 320_000,
    16: 580_000,
    32: 1_050_000,
    64: 1_800_000,
}

efficiencies = compute_scaling_efficiency(hypothetical_throughputs, baseline_gpus=1)

print("\nScaling Efficiency (hypothetical FSDP run):")
print(f"{'GPUs':>6} {'Tok/s':>12} {'Speedup':>10} {'Efficiency':>12}")
print("-" * 44)
base = hypothetical_throughputs[1]
for gpus in sorted(hypothetical_throughputs):
    tps = hypothetical_throughputs[gpus]
    speedup = tps / base
    eff = efficiencies[gpus]
    print(f"{gpus:>6} {tps:>12,} {speedup:>9.1f}x {eff:>11.1%}")

if HAS_MPL:
    gpus = sorted(hypothetical_throughputs.keys())
    effs = [efficiencies[g] for g in gpus]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(gpus, effs, "o-", color="steelblue", linewidth=2, markersize=8)
    ax.axhline(y=1.0, color="green", linestyle="--", alpha=0.5, label="Perfect scaling")
    ax.set_xlabel("Number of GPUs")
    ax.set_ylabel("Scaling Efficiency")
    ax.set_title("FSDP Scaling Efficiency (Hypothetical)")
    ax.set_xscale("log", base=2)
    ax.set_xticks(gpus)
    ax.set_xticklabels([str(g) for g in gpus])
    ax.set_ylim(0.5, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("scaling_efficiency.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved scaling_efficiency.png")
else:
    print("Install matplotlib to see the scaling chart.")

## Summary

Key takeaways:

- **ZeRO/FSDP is the practical default**: Minimal code changes from DDP, and Stage 3 reduces per-GPU memory by $N\times$ where $N$ is the GPU count.
- **Memory math matters**: A 7B model needs ~104 GB for DDP but only ~1.6 GB per GPU with ZeRO-3 on 64 GPUs. Understanding these numbers prevents OOM surprises.
- **Pipeline parallelism has a cost**: The bubble fraction wastes 10–30% of compute depending on the stage/micro-batch ratio. More micro-batches help but increase memory.
- **Scaling efficiency degrades gradually**: FSDP typically achieves 90–95% efficiency up to 64 GPUs, dropping to 80–85% at 256 GPUs as communication overhead grows.
- **Profile early**: Throughput measurement catches configuration mistakes (wrong batch size, missing overlap) before they waste expensive GPU hours.